# 75 — Responder Phase 0: Gemini-judge offline harness

nDCG is capped (~0.29 Blind-A); the LLM axis (0.30 composite weight, 2.7/5,
untouched) is the biggest remaining lever. This notebook scores the responder
OFFLINE with the real Gemini API (Personalization + Explanation Quality) on a dev
subset, so we can A/B responder changes (top_n_for_prompt, length, prompt) without
burning Blind-A submissions.

Run order: cell 1 (setup) -> cell 2 (baseline inference, config 194 subset) ->
cell 3 (Gemini judge). Needs GEMINI_API_KEY in Colab secrets.

In [ ]:
# 1) Setup. Clone repo, install deps, mount Drive, set GEMINI_API_KEY from secrets.
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# Gemini key — REQUIRED for cell 3. Add it to Colab secrets as GEMINI_API_KEY.
try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('[setup] GEMINI_API_KEY loaded from Colab secrets')
except Exception:
    print('[setup] WARNING: GEMINI_API_KEY not in Colab secrets — cell 3 will fail until you add it')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'recall-union-lgbm'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

# Symlink caches (same as nb74) so the retriever/reranker/SASRec load from Drive.
DRIVE_BASE='/content/drive/MyDrive'; LOCAL_BASE='/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, sub in [('retrieval_v2','recsys2026_retrieval_v2_cache'), ('dense','recsys2026_dense_cache')]:
    src=f'{DRIVE_BASE}/{sub}'; dst=f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11' 'datasets' \
    'pandas<3.0' 'sentence-transformers>=3.0' 'lightgbm' 'omegaconf' 'bm25s' 'FlagEmbedding>=1.3' \
    'google-generativeai'

## Baseline responder inference on a dev subset (config 194)

Runs the SHIPPED pipeline (config 194: union+SASRec -> lgbm_clean_full -> v5-kto
responder, top_n_for_prompt=1) on the first 50 dev sessions (=400 turns) and writes
exp/inference/devset/194-...json. This is the BASELINE the judge scores; later
Phase-1 A/Bs re-run this cell with a different config tid.

In [ ]:
# 2) Baseline inference on 50 dev sessions (~400 turns). ~15-30 min.
TID = '194-union-sasrec-lgbm-cleanfull-v5kto-blindA'
SUBSET = 50  # sessions; 50*8=400 turns. Bump later for tighter judge estimates.
%cd /content/recsys2026/music-crs-baselines
!python run_inference_devset.py --tid {TID} --subset {SUBSET} --batch_size 8 2>&1 | tail -30
%cd /content/recsys2026
import json
preds = json.load(open(f'music-crs-baselines/exp/inference/devset/{TID}.json'))
print('[infer] rows:', len(preds), '| sample response:', preds[0]['predicted_response'][:160])

## Gemini judge: score the baseline responses

Calls the real Gemini API on Personalization + Explanation Quality (0-5 each).
Prints the LLM-axis proxy = mean of the two. This is the number Phase-1 changes
must beat. RECONCILE the judge rubric (scripts/gemini_judge_responses.py
JUDGE_INSTRUCTIONS) with the official challenge rubric when available.

In [ ]:
# 3) Judge the baseline. Needs GEMINI_API_KEY (cell 1). --limit caps cost.
TID = '194-union-sasrec-lgbm-cleanfull-v5kto-blindA'
%cd /content/recsys2026/music-crs-baselines
!python ../scripts/gemini_judge_responses.py --tid {TID} --limit 400 --sleep 0.5 2>&1 | tail -20
%cd /content/recsys2026